# 03 · Mobility, resistance, and swimming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fairyflies9/fluidlets/blob/main/tutorials/03_mobility_resistance_swimming.ipynb)

Regularized Stokeslets turn Stokes flow into linear algebra. If $\mathbf F$ collects all the
point forces and $\mathbf U$ all the point velocities, then

$$
\mathbf U = \frac{1}{\mu}\,\mathsf K\,\mathbf F ,
$$

where $\mathsf K$ is the $3N\times3N$ matrix of $3\times3$ blocks $H_1\mathbf I + H_2\,\mathbf d\mathbf d^T$
from [tutorial 01](01_blob_to_stokeslet.ipynb). Every problem in this notebook is one way of using
that equation:

| problem | known | unknown | Fluidlets |
|---|---|---|---|
| **mobility** | forces | velocities | `fl.evaluate` (a matrix-vector product) |
| **resistance** | velocities | forces | `fl.resistance` (a linear solve) |
| **mixed** | some of each, plus constraints | the rest | build the system with `fl.assemble` |
| **free swimming** | shape changes, zero net force and torque | forces, body velocity | `fluidlets.swimming` |

Along the way we add a no-slip wall with `fl.WallStokeslet3D` and check it against Brenner's
exact drag on a sphere approaching a wall.

In [ ]:
# On Colab or Binder, install Fluidlets first (skipped when it is already installed).
import importlib.util, subprocess, sys
if importlib.util.find_spec("fluidlets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/fairyflies9/fluidlets"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fluidlets as fl
from fluidlets.analytic import brenner_wall_drag, translating_sphere_velocity
from fluidlets.geometry import cube_sphere, ring_sphere

kernel = fl.Stokeslet3D()
mu, a = 1.0, 1.0
ez = np.array([0.0, 0.0, 1.0])

## 1. The mobility problem: forces in, velocities out

A classical fact from Stokes (1851): the traction on a translating sphere is **uniform** over its
surface. So if we spread a total force $\mathbf F$ evenly over $N$ surface points, every point
should move with the same velocity, $\mathbf U = \mathbf F/(6\pi\mu a)$, and the flow outside
should be the exact translating-sphere flow.

This is a mobility problem: no solve, just one call to `fl.evaluate`.

In [ ]:
n, eps = 12, 0.05                                   # eps ~ h/3, see tutorial 02
X = cube_sphere(n, a)
N = len(X)
F = 6 * np.pi * mu * a * ez                         # should give U = e_z
forces = np.tile(F / N, (N, 1))

u_surface = fl.evaluate(kernel, X, X, forces, eps, mu)
print(f"N = {N} points; surface velocity: mean = {u_surface.mean(axis=0).round(4)}, "
      f"std of u_z = {u_surface[:, 2].std():.4f}")

In [ ]:
g = np.linspace(-3, 3, 61)
Xg, Zg = np.meshgrid(g, g)
grid = np.c_[Xg.ravel(), 0 * Xg.ravel(), Zg.ravel()]
u = fl.evaluate(kernel, grid, X, forces, eps, mu)
exact = translating_sphere_velocity(grid, ez, a)
err = np.linalg.norm(u - exact, axis=1).reshape(Xg.shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
ux, uz = u[:, 0].reshape(Xg.shape), u[:, 2].reshape(Xg.shape)
axs[0].streamplot(g, g, ux, uz, density=1.3, color=np.hypot(ux, uz), cmap="viridis")
axs[0].set_title("lab frame: sphere moving up")
pc = axs[1].pcolormesh(Xg, Zg, np.log10(err + 1e-16), shading="auto", vmin=-4, vmax=-1)
fig.colorbar(pc, ax=axs[1], label="log10 |u - exact|")
axs[1].set_title("error vs exact translating sphere")
for ax in axs:
    ax.add_patch(plt.Circle((0, 0), a, fill=False, color="k"))
    ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("z")
plt.tight_layout(); plt.show()

The surface moves at $\mathbf U\approx\mathbf e_z$ to within about 1 %, and the flow matches the
exact solution closely away from the surface. The error is largest within a few $\varepsilon$ of
the points, as expected from tutorials 01 and 02.

## 2. The resistance problem: velocities in, forces out

Usually we know how a body moves and want the force. Then we must **solve** $\mathsf K\mathbf F = \mu\mathbf U$.
`fl.resistance` does this (dense LU for small problems, GMRES for large ones).

In [ ]:
U = np.tile(ez, (N, 1))
forces_res, info = fl.resistance(kernel, X, U, eps, mu)
drag = forces_res.sum(axis=0)
print(f"total force = {drag.round(4)},  exact 6 pi = {6 * np.pi:.4f}")
print(f"force per point: mean {forces_res[:, 2].mean():.5f}, std {forces_res[:, 2].std():.5f}  (uniform, as Stokes said)")
print(info)

### The matrix behind both problems

`fl.assemble` returns $\mathsf K/\mu$ itself. Mobility is multiplication by it; resistance is
solving with it. The Lorentz reciprocal theorem says the mobility operator is symmetric and
positive definite (pushing on the fluid always does positive work), and the regularized matrix
inherits both properties. Let us check on a small sphere.

In [ ]:
Xs = cube_sphere(6, a)
K = fl.assemble(kernel, Xs, Xs, 0.1, mu)
f_test = np.random.default_rng(0).normal(size=(len(Xs), 3))
print("shape:", K.shape)
print("K f == evaluate(f):", np.allclose(K @ f_test.ravel(), fl.evaluate(kernel, Xs, Xs, f_test, 0.1, mu).ravel()))
print("symmetric:", np.allclose(K, K.T))
print(f"smallest eigenvalue: {np.linalg.eigvalsh(K).min():.3e} (> 0)")

## 3. A mixed problem: a sedimenting sphere

Now suppose we know the **total** force (gravity minus buoyancy), but neither the force
distribution nor the velocity. The sphere moves rigidly with an unknown $\mathbf U$, so the
unknowns are the $N$ forces and $\mathbf U$, and the equations are

$$
\frac1\mu\sum_j \mathsf K_{kj}\,\mathbf f_j - \mathbf U = \mathbf 0 \quad (k = 1..N), \qquad
\sum_j \mathbf f_j = \mathbf F .
$$

That is a $(3N+3)\times(3N+3)$ system, which we build from `fl.assemble`:

$$
\begin{pmatrix} \mathsf K/\mu & -\mathsf I_{\text{stack}} \\ \mathsf I_{\text{stack}}^T & 0 \end{pmatrix}
\begin{pmatrix} \mathbf F_{\text{points}} \\ \mathbf U \end{pmatrix}
=
\begin{pmatrix} \mathbf 0 \\ \mathbf F \end{pmatrix},
$$

where $\mathsf I_{\text{stack}}$ is $N$ identity blocks stacked vertically.

In [ ]:
F_gravity = -6 * np.pi * mu * a * ez                 # chosen so that the exact U = -e_z
stack = np.tile(np.eye(3), (N, 1))                   # (3N, 3)
A = np.zeros((3 * N + 3, 3 * N + 3))
A[:3 * N, :3 * N] = fl.assemble(kernel, X, X, eps, mu)
A[:3 * N, 3 * N:] = -stack
A[3 * N:, :3 * N] = stack.T
rhs = np.concatenate([np.zeros(3 * N), F_gravity])
sol = np.linalg.solve(A, rhs)
print(f"settling velocity U = {sol[3 * N:].round(4)}  (Stokes: {F_gravity / (6 * np.pi * mu * a)})")

The free-swimming problem in Section 5 has exactly this structure, with zero total force and
torque and a prescribed surface motion on the right-hand side.

## 4. Walls: the regularized image system

Real cells swim near coverslips, epithelia and the walls of channels. A wall at $z = 0$ slows
everything nearby down. `fl.WallStokeslet3D` adds to each Stokeslet a system of images (a
Stokeslet, a dipole, a doublet and a rotlet at the mirror point) chosen by Ainley et al.,
J. Comput. Phys. **227**, 4600 (2008), so that the velocity vanishes on the wall. It is used
exactly like the free-space kernel. First check the no-slip condition:

In [ ]:
wall = fl.WallStokeslet3D()
source, f0 = np.array([[0.2, -0.1, 0.5]]), np.array([[1.0, -0.3, 0.7]])
on_wall = np.c_[np.random.default_rng(1).uniform(-2, 2, (1000, 2)), np.zeros(1000)]
print("max |u| on z = 0, free space:", np.abs(fl.evaluate(kernel, on_wall, source, f0, 0.05)).max())
print("max |u| on z = 0, with wall: ", np.abs(fl.evaluate(wall, on_wall, source, f0, 0.05)).max())

### Drag on a sphere approaching a wall

Brenner (1961) found the exact drag on a sphere of radius $a$ moving towards a wall with its centre
at height $d$. We repeat the test of Ainley et al. (Sec. 5.4, Table 2): a sphere of radius 0.1 with
`ring_sphere(19)` (468 points), blob width $\varepsilon = 0.4\,h^{0.9}$, and the drag divided by
Stokes' $6\pi\mu aU$.

In [ ]:
a_w = 0.1
unit = ring_sphere(19, a_w)
S = len(unit)
h = np.sqrt(4 * np.pi * a_w**2 / S)
eps_w = 0.4 * h**0.9                                  # Ainley et al.'s choice for normal motion

d_over_a = np.cosh([1.0, 0.5, 0.3])                   # Ainley et al.'s rows alpha = 1, 0.5, 0.3
d_over_a = np.r_[d_over_a, 2.0, 3.0, 5.0]
drag_wall = []
for d in d_over_a:
    Xw = unit + [0, 0, d * a_w]
    fw, info = fl.resistance(wall, Xw, np.tile(ez, (S, 1)), eps_w, mu, method="gmres", rtol=1e-8, maxiter=1000)
    drag_wall.append(fw[:, 2].sum() / (6 * np.pi * mu * a_w))
drag_wall = np.array(drag_wall)
f_free, _ = fl.resistance(kernel, unit, np.tile(ez, (S, 1)), eps_w, mu)
print(f"S = {S}, h = {h:.4f}, eps = {eps_w:.4f}; free-space drag / 6 pi mu a U = {f_free[:, 2].sum() / (6 * np.pi * a_w):.4f}\n")
print("   d/a    computed   Brenner")
for d, F in zip(d_over_a, drag_wall):
    print(f"{d:7.4f} {F:9.4f} {brenner_wall_drag(d)[0]:9.4f}")

In [ ]:
dd = np.linspace(1.15, 6, 200)
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(dd, brenner_wall_drag(dd), "k-", label="Brenner (1961), exact")
ax.plot(d_over_a, drag_wall, "o", label=f"WallStokeslet3D, S = {S}")
ax.axhline(1, color="gray", ls=":", label="no wall")
ax.set_xlabel("d / a  (centre height / radius)"); ax.set_ylabel(r"drag / $6\pi\mu a U$")
ax.legend(); plt.show()

The wall roughly doubles the drag at one diameter from the wall, and far from it the regularized
solution follows Brenner's curve within a few percent. Close to contact it degrades: about 7 % too
high at $d/a = 1.54$ and 11 % too high at $1.13$ (Ainley et al. report the same 3.2441 and 10.2872 at
this resolution), and then 17 % too *low* at $d/a = 1.045$. There the gap between sphere and wall,
$0.0045$, is narrower than $\varepsilon\approx0.0099$: the blobs on the sphere overlap the wall, and the
lubrication flow in the gap, which is what produces the huge exact drag, cannot be represented.
The gap must be resolved by the points **and** be wider than $\varepsilon$, the lesson of tutorial 02
again. Refining to `ring_sphere(25)` or `ring_sphere(34)` improves the close-contact rows.

## 5. Free swimming

A swimmer has no external force or torque on it. It moves its surface (beats a flagellum, waves
cilia), and the fluid pushes back so that the whole body moves. In the body frame the surface
points $\boldsymbol\xi_m$ have prescribed velocities $\dot{\boldsymbol\xi}_m$ (the *beat*). In the
lab the body frame moves with unknown $\mathbf U$ and rotates with unknown $\boldsymbol\Omega$, so the
no-slip condition, force balance and torque balance read

$$
\frac1\mu\sum_n \mathsf K_{mn}\,\mathbf f_n - \mathbf U - \boldsymbol\Omega\times(\mathbf x_m - \mathbf x_0) = \dot{\boldsymbol\xi}_m,
\qquad \sum_n \mathbf f_n = \mathbf 0, \qquad \sum_n \mathbf x_n\times\mathbf f_n = \mathbf 0 .
$$

It is the sedimentation system of Section 3 with three more unknowns ($\boldsymbol\Omega$), zero on
the right of the balances, and the beat as the forcing. `fluidlets.swimming.swimming_problem`
builds and solves it, following Gallagher & Smith, Phys. Rev. Fluids **3**, 053101 (2018), who
also integrate the kernel over a finer set of quadrature points `X` than the force points `x`.

### A test with an exact answer: the uniform-slip sphere

Let every surface point move backwards, $\dot{\boldsymbol\xi}_m = -\mathbf V$, relative to the body.
Then $\mathbf f = \mathbf 0$, $\mathbf U = \mathbf V$, $\boldsymbol\Omega = \mathbf 0$ satisfies all
the equations exactly: the body moves forward at $\mathbf V$, the surface moves backward relative to
it at $\mathbf V$, so in the lab the surface is at rest and the fluid is never disturbed. Since the
system is nonsingular this is *the* solution, for any $\varepsilon$ and any discretization, so the
solver must reproduce it to roundoff.

In [ ]:
from fluidlets.swimming import SwimmerShape, swim, swimming_problem

x_body = cube_sphere(4, a)                 # force points
X_quad = cube_sphere(10, a)                # finer quadrature points
V = np.array([0.2, -0.1, 0.7])
shape = SwimmerShape(x=x_body, v=-np.tile(V, (len(x_body), 1)), X=X_quad)

state = swimming_problem(shape, x0=np.zeros(3), B=np.eye(3), eps=0.05, mu=mu)
print("U     =", state.U, "  (V =", V, ")")
print("Omega =", state.Omega)
print("max |f| =", np.abs(state.forces).max())
assert np.allclose(state.U, V, atol=1e-12) and np.abs(state.forces).max() < 1e-10

### Following a swimmer in time

Add a body-frame spin $\mathbf W$ to the slip, $\dot{\boldsymbol\xi} = -(\mathbf V + \mathbf W\times\boldsymbol\xi)$.
The swimmer then moves with $\mathbf V$ and turns with $\mathbf W$, both measured in its own frame.
If $\mathbf W\perp\mathbf V$ it swims in a circle of radius $|\mathbf V|/|\mathbf W|$.
`fluidlets.swimming.swim` integrates $\dot{\mathbf x}_0 = \mathbf U$ and
$\dot{\mathbf b}_j = \boldsymbol\Omega\times\mathbf b_j$ for the body axes $\mathbf b_j$, solving the
swimming problem at every step.

In [ ]:
V_body = np.array([0.0, 0.0, 0.5])         # swim along the body z-axis
W_body = np.array([0.25, 0.0, 0.0])        # and turn about the body x-axis

def shape_at(t):
    x = cube_sphere(3, a)
    return SwimmerShape(x=x, v=-(V_body + np.cross(W_body, x)), X=cube_sphere(6, a))

period = 2 * np.pi / np.linalg.norm(W_body)
# Every right-hand-side evaluation of the ODE is one swimming solve, so a modest
# tolerance keeps this cell fast (a few seconds).
traj = swim(shape_at, (0.0, period), eps=0.1, mu=mu, t_eval=np.linspace(0, period, 60),
            rtol=1e-4, atol=1e-6)
radius = np.linalg.norm(V_body) / np.linalg.norm(W_body)
centre = np.array([0.0, -radius, 0.0])
dist = np.linalg.norm(traj.x0 - centre, axis=1)
print(f"predicted radius {radius}; distance from the centre stays in [{dist.min():.4f}, {dist.max():.4f}]")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(traj.x0[:, 1], traj.x0[:, 2], ".-")
ax.quiver(traj.x0[::6, 1], traj.x0[::6, 2], traj.B[::6, 1, 2], traj.B[::6, 2, 2], color="r",
          scale=6, width=0.012, label="body z-axis")
ax.set_aspect("equal"); ax.set_xlabel("y"); ax.set_ylabel("z"); ax.legend()
ax.set_title("A spinning slip swimmer swims in a circle"); plt.show()

These two cases have exact answers, which makes them good tests, but the fluid is never disturbed
in either of them. Real swimmers push fluid around; the first exercise below is a classic model
swimmer, the *squirmer*, whose speed is not obvious. The example `examples/biflagellate_swimmer`
applies the same solver to a cell with two beating flagella.

## Summary

* $\mathbf U = \mathsf K\mathbf F/\mu$. Mobility: `fl.evaluate`. Resistance: `fl.resistance`. Mixed and
  swimming problems: add unknowns and constraints around `fl.assemble`.
* $\mathsf K$ is symmetric positive definite, as reciprocity demands.
* `fl.WallStokeslet3D` enforces no slip on $z = 0$; it reproduces Brenner's wall drag.
* The free-swimming problem adds $\mathbf U$, $\boldsymbol\Omega$ and the force- and torque-free
  conditions; a uniform-slip sphere swims at exactly $\mathbf V$.

## Exercises

1. **A squirmer.** Lighthill (1952) and Blake (1971) modelled ciliates as spheres with tangential
   surface slip $\mathbf u_s = B_1\sin\theta\,\mathbf e_\theta$, where $\theta$ is the angle from
   the swimming direction $\mathbf e_z$ and $\mathbf e_\theta = (\cos\theta\cos\varphi, \cos\theta\sin\varphi, -\sin\theta)$.
   Use it as the beat `v` in `swimming_problem` and show that the swimmer moves at about
   $U = \tfrac23 B_1$. How does the answer change as you refine the force points, the quadrature
   points and $\varepsilon$? Plot the flow around it in the lab frame and in the swimmer's frame.
2. **Swimming near a wall.** Pass `kernel=fl.WallStokeslet3D()` to `swimming_problem` and put the
   uniform-slip sphere at height $z = 2a$. Does it still swim at exactly $\mathbf V$? Explain why
   with the argument of Section 5. Now repeat with the squirmer swimming parallel to the wall. What changes?
3. **Parallel to the wall.** Repeat Section 4 for motion parallel to the wall (velocity $\mathbf e_x$,
   $\varepsilon = 0.22\,h^{0.9}$) and compare with O'Neill's (1964) values 1.0591, 1.1738, 1.5675,
   2.1515 at $d/a = \cosh 3, \cosh 2, \cosh 1, \cosh 0.5$. Is a wall more important for motion
   towards it or along it?
4. **A falling prolate spheroid.** Stretch the sphere into a spheroid (multiply the $z$-coordinates
   of `cube_sphere` by 2, and choose $\varepsilon$ from the new spacing). Solve the sedimentation
   problem of Section 3 with the force along $\mathbf e_z$, along $\mathbf e_x$, and along
   $(\mathbf e_x + \mathbf e_z)/\sqrt2$. In the last case, which way does the body move? Why does it
   not fall straight down?
5. **Mobility from resistance.** Take the forces from Section 2 and evaluate the velocity at the
   force points with `fl.evaluate`. Why do you recover $\mathbf U = \mathbf e_z$ to roundoff, while
   the uniform forces of Section 1 only gave it to about 1 %?